# VGG16 + Random Forest — Augmented Dataset
Workflow: Nạp dữ liệu → Trích đặc trưng VGG16 → Huấn luyện RF → Đánh giá kết quả → Xuất model

## 1. Cài đặt & Nhập thư viện

In [19]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Input, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
import joblib

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow version : 2.19.0
GPU available      : True


## 2. Cấu hình đường dẫn (Kaggle)

In [10]:
# ── Kaggle: dataset được mount sẵn tại /kaggle/input/ ──
DATASET_ROOT = '/kaggle/input/datasets/trnnguynlmhuy/original-dataset'   # <-- thay bằng tên dataset Kaggle của bạn

DATA_SPLITS = {
    'train' : os.path.join(DATASET_ROOT, 'train'),
    'val'   : os.path.join(DATASET_ROOT, 'val'),
    'test'  : os.path.join(DATASET_ROOT, 'test'),
}

OUTPUT_DIR = '/kaggle/working/saved_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for split, path in DATA_SPLITS.items():
    exists = os.path.isdir(path)
    print(f'  [{split}] {path}  →  {"OK" if exists else "NOT FOUND"}')

  [train] /kaggle/input/datasets/trnnguynlmhuy/original-dataset/train  →  OK
  [val] /kaggle/input/datasets/trnnguynlmhuy/original-dataset/val  →  OK
  [test] /kaggle/input/datasets/trnnguynlmhuy/original-dataset/test  →  OK


## 3. Nạp Dataset

In [11]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 2024

def load_split(directory, shuffle=False):
    """Trả về tf.data.Dataset từ thư mục ảnh có cấu trúc class-subfolder."""
    ds = image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='int',
        shuffle=shuffle,
        seed=SEED,
    )
    return ds

train_ds = load_split(DATA_SPLITS['train'], shuffle=False)
val_ds   = load_split(DATA_SPLITS['val'],   shuffle=False)
test_ds  = load_split(DATA_SPLITS['test'],  shuffle=False)

CLASS_NAMES  = train_ds.class_names
NUM_CLASSES  = len(CLASS_NAMES)
print(f'Số lớp phân loại : {NUM_CLASSES}')
print(f'Tên lớp          : {CLASS_NAMES[:5]} ...')

Found 20080 files belonging to 150 classes.
Found 4248 files belonging to 150 classes.
Found 4468 files belonging to 150 classes.
Số lớp phân loại : 150
Tên lớp          : ['ABBOTTS BOOBY', 'ABYSSINIAN GROUND HORNBILL', 'AFRICAN PIED HORNBILL', 'AFRICAN PYGMY GOOSE', 'ALPINE CHOUGH'] ...


## 4. Xây dựng Bộ Trích Đặc Trưng VGG16

In [12]:
def build_vgg16_extractor(img_shape=(224, 224, 3)):
    """
    VGG16 (ImageNet, không top) + GlobalAveragePooling2D.
    Toàn bộ trọng số bị đóng băng → chỉ dùng để trích đặc trưng.
    Output: vector 512 chiều mỗi ảnh.
    """
    inp = Input(shape=img_shape)
    base = VGG16(
        include_top=False,
        weights='imagenet',
        input_tensor=inp
    )
    base.trainable = False          # đóng băng hoàn toàn
    gap = GlobalAveragePooling2D()(base.output)
    extractor = Model(inputs=inp, outputs=gap, name='vgg16_gap_extractor')
    return extractor

feature_extractor = build_vgg16_extractor()
print(f'Feature vector dim   : {feature_extractor.output_shape}')   # (None, 512)
print(f'Trainable parameters : {feature_extractor.count_params()} (phải = 0 params trainable)')
feature_extractor.summary()

Feature vector dim   : (None, 512)
Trainable parameters : 14714688 (phải = 0 params trainable)


Model: "vgg16_gap_extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

## 5. Trích Xuất Đặc Trưng từ Toàn Bộ Tập Dữ Liệu

In [13]:
def extract_features_from_dataset(extractor, dataset, split_name=''):
    """
    Duyệt qua dataset → áp dụng VGG16 preprocessing → predict features.
    Trả về (features: np.ndarray, labels: np.ndarray).
    """
    print(f'Trích đặc trưng — tập [{split_name}] ...')
    all_feats, all_labels = [], []
    for batch_imgs, batch_lbls in dataset:
        preprocessed = preprocess_input(tf.cast(batch_imgs, tf.float32))
        feats = extractor(preprocessed, training=False).numpy()
        all_feats.append(feats)
        all_labels.append(batch_lbls.numpy())
    X = np.concatenate(all_feats,  axis=0)
    y = np.concatenate(all_labels, axis=0)
    print(f'  features shape: {X.shape} | labels shape: {y.shape}')
    return X, y

X_train, y_train = extract_features_from_dataset(feature_extractor, train_ds, 'train')
X_val,   y_val   = extract_features_from_dataset(feature_extractor, val_ds,   'val')
X_test,  y_test  = extract_features_from_dataset(feature_extractor, test_ds,  'test')

# Gộp train + val để tối đa dữ liệu huấn luyện RF
X_tv = np.concatenate([X_train, X_val], axis=0)
y_tv = np.concatenate([y_train, y_val], axis=0)
print(f'\nTổng train+val : {X_tv.shape}')
print(f'Test set       : {X_test.shape}')

# Kiểm tra phân phối đặc trưng
print(f'\n--- Thống kê X_train+val ---')
print(f'Mean  : {X_tv.mean():.4f}')
print(f'Std   : {X_tv.std():.4f}')
print(f'Min   : {X_tv.min():.4f}')
print(f'Max   : {X_tv.max():.4f}')
print(f'% < 0 : {(X_tv < 0).mean()*100:.2f}%')

Trích đặc trưng — tập [train] ...
  features shape: (20080, 512) | labels shape: (20080,)
Trích đặc trưng — tập [val] ...
  features shape: (4248, 512) | labels shape: (4248,)
Trích đặc trưng — tập [test] ...
  features shape: (4468, 512) | labels shape: (4468,)

Tổng train+val : (24328, 512)
Test set       : (4468, 512)

--- Thống kê X_train+val ---
Mean  : 3.3637
Std   : 5.4659
Min   : 0.0000
Max   : 160.2202
% < 0 : 0.00%


## 6. Huấn Luyện Random Forest

In [22]:
# ============================================================
# Grid Search Random Forest trên vector đặc trưng VGG16
# ============================================================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt"],
    "bootstrap": [True]
}

rf = RandomForestClassifier(
    n_jobs=-1,
    random_state=42
)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    verbose=3,
    n_jobs=-1,
)

print("Bắt đầu chạy Grid Search...")
grid.fit(X_tv, y_tv)
print("Hoàn thành Grid Search!")

print("\nBest params:")
print(grid.best_params_)

print(f"\nBest CV accuracy: {grid.best_score_:.4f}")

Bắt đầu chạy Grid Search...
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV 2/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.719 total time= 2.1min
[CV 3/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=0.740 total time= 4.1min
[CV 2/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200;, score=0.760 total time= 4.0min


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.714 total time= 2.0min
[CV 2/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=0.751 total time= 4.1min
[CV 1/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200;, score=0.744 total time= 4.0min
[CV 3/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=2, n_estimators=100;, score=0.712 total time= 2.0min
[CV 1/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.709 total time= 2.1min
[CV 1/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100;, score=0.711 total time= 2.1min
[CV 2/3] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1

## 7. Train random forest với bộ tham số tốt nhất tìm được sau khi chạy gridsearch

In [23]:
# ============================================================
# Train Random Forest cuối cùng với best params
# ============================================================

best_params = grid.best_params_

best_rf = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=42
)

print("Bắt đầu train Random Forest với best params...")
best_rf.fit(X_tv, y_tv)
print("Hoàn thành train model cuối!")

Bắt đầu train Random Forest với best params...
Hoàn thành train model cuối!


In [26]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================
# Predict
# =========================
y_train_pred = best_rf.predict(X_tv)
y_test_pred = best_rf.predict(X_test)

# =========================
# Accuracy
# =========================
train_acc = accuracy_score(y_tv, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")

# =========================
# Classification Report
# =========================
print("\n=== Train Classification Report ===")
print(classification_report(y_tv, y_train_pred))

print("\n=== Test Classification Report ===")
print(classification_report(y_test, y_test_pred))

# =========================
# Confusion Matrix
# =========================
print("\n=== Confusion Matrix (Test) ===")
print(confusion_matrix(y_test, y_test_pred))

Train Accuracy: 0.9998
Test Accuracy:  0.8559

=== Train Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       156
           1       1.00      1.00      1.00       155
           2       1.00      1.00      1.00       162
           3       1.00      1.00      1.00       155
           4       1.00      1.00      1.00       144
           5       1.00      1.00      1.00       155
           6       1.00      1.00      1.00       148
           7       1.00      1.00      1.00       163
           8       1.00      0.99      1.00       155
           9       1.00      1.00      1.00       162
          10       0.99      1.00      0.99       163
          11       1.00      1.00      1.00       187
          12       1.00      1.00      1.00       165
          13       1.00      1.00      1.00       162
          14       1.00      1.00      1.00       165
          15       1.00      1.00      1.00       14

## 8. Lưu Model

In [ ]:
rf_path = os.path.join(OUTPUT_DIR, 'vgg16_rf_aug.joblib')
joblib.dump(clf, rf_path)
print(f'Model đã được lưu tại: {rf_path}')
print(f'Kích thước file       : {os.path.getsize(rf_path)/1e6:.1f} MB')